In [1]:

import re


def is_variable(term):
    return term[0].islower()


def parse_literal(text):
    text = text.strip()

    negated = False

    if text.startswith("~"):
        negated = True
        text = text[1:].strip()
    elif text.upper().startswith("NOT "):
        negated = True
        text = text[4:].strip()

    match = re.fullmatch(r"([A-Za-z][A-Za-z0-9_]*)\s*\((.*)\)", text)

    if not match:
        raise ValueError("Invalid literal: " + text)

    predicate = match.group(1)
    arguments = match.group(2).strip()

    if arguments:
        arguments = [x.strip() for x in arguments.split(",")]
    else:
        arguments = []

    return (negated, predicate, tuple(arguments))


def parse_clause(text):
    parts = re.split(r"\s*(?:\||\bOR\b)\s*", text, flags=re.I)
    return tuple(parse_literal(part) for part in parts)


def display_literal(literal):
    neg, predicate, args = literal
    prefix = "~" if neg else ""
    return prefix + predicate + "(" + ",".join(args) + ")"


def display_clause(clause):
    if not clause:
        return "{}"
    return " OR ".join(display_literal(x) for x in clause)


def substitute(term, substitutions):
    while term in substitutions and substitutions[term] != term:
        term = substitutions[term]
    return term


def unify_terms(a, b, substitutions):

    a = substitute(a, substitutions)
    b = substitute(b, substitutions)

    if a == b:
        return True

    if is_variable(a):
        substitutions[a] = b
        return True

    if is_variable(b):
        substitutions[b] = a
        return True

    return False


def unify_arguments(args1, args2):

    if len(args1) != len(args2):
        return None

    substitutions = {}

    for a, b in zip(args1, args2):
        if not unify_terms(a, b, substitutions):
            return None

    return substitutions


def apply_substitution(literal, substitutions):

    neg, predicate, args = literal

    new_args = tuple(
        substitute(arg, substitutions) for arg in args
    )

    return (neg, predicate, new_args)


def standardize_clause(clause, suffix):

    variables = {}

    new_clause = []

    for neg, predicate, args in clause:

        new_args = []

        for arg in args:

            if is_variable(arg):

                if arg not in variables:
                    variables[arg] = arg + "_" + str(suffix)

                new_args.append(variables[arg])

            else:
                new_args.append(arg)

        new_clause.append((neg, predicate, tuple(new_args)))

    return tuple(new_clause)


def resolve_clauses(clause1, clause2, suffix):

    clause1 = standardize_clause(clause1, suffix)
    clause2 = standardize_clause(clause2, suffix + 1)

    resolvents = []

    for i, lit1 in enumerate(clause1):

        for j, lit2 in enumerate(clause2):

            neg1, pred1, args1 = lit1
            neg2, pred2, args2 = lit2

            if pred1 != pred2 or neg1 == neg2:
                continue

            substitutions = unify_arguments(args1, args2)

            if substitutions is None:
                continue

            remaining = (
                clause1[:i] + clause1[i + 1:] +
                clause2[:j] + clause2[j + 1:]
            )

            new_clause = tuple(
                apply_substitution(lit, substitutions)
                for lit in remaining
            )

            resolvents.append(new_clause)

    return resolvents


def clause_key(clause):

    variable_names = {}

    def normalize(term):

        if is_variable(term):

            if term not in variable_names:
                variable_names[term] = "v" + str(len(variable_names))

            return variable_names[term]

        return term

    literals = sorted(clause, key=lambda x: (x[1], x[0], x[2]))

    result = []

    for neg, predicate, args in literals:

        new_args = tuple(normalize(arg) for arg in args)

        result.append((neg, predicate, new_args))

    return tuple(result)


def resolution(clauses):

    known = list(clauses)
    known_keys = {clause_key(c) for c in known}

    print("\nInitial Clauses:")

    for i, clause in enumerate(known, 1):
        print("C" + str(i) + ":", display_clause(clause))

    print("\nResolution Process:")

    step = 1
    suffix = 1
    max_steps = 500

    while step <= max_steps:

        new_clauses = []

        for i in range(len(known)):

            for j in range(i + 1, len(known)):

                results = resolve_clauses(
                    known[i], known[j], suffix
                )

                suffix += 2

                for result in results:

                    key = clause_key(result)

                    if key in known_keys:
                        continue

                    print("\nStep", step)
                    print("Clause 1:", display_clause(known[i]))
                    print("Clause 2:", display_clause(known[j]))
                    print("New Clause:", display_clause(result))

                    if not result:
                        print("\nEmpty Clause Obtained!")
                        print("Query is PROVED!")
                        return True

                    known_keys.add(key)
                    new_clauses.append(result)

                    step += 1

                    if step > max_steps:
                        break

                if step > max_steps:
                    break

            if step > max_steps:
                break

        if not new_clauses:
            print("\nNo more clauses can be resolved.")
            print("Query could not be proved.")
            return False

        known.extend(new_clauses)

    print("\nResolution step limit reached.")
    print("Proof not established within the limit.")
    return False


def main():

    print("FIRST ORDER LOGIC - RESOLUTION")
    print("--------------------------------")

    clauses = []

    n = int(input("\nEnter number of CNF clauses: "))

    print("\nEnter clauses using | for OR.")
    print("Use ~ or NOT for negation.")
    print("Example: ~Food(x) | Likes(John,x)")

    for i in range(n):

        text = input("Enter clause " + str(i + 1) + ": ")

        try:
            clauses.append(parse_clause(text))
        except ValueError as error:
            print(error)
            return

    print("\nStarting Resolution...")

    resolution(clauses)


if __name__ == "__main__":
    main()


FIRST ORDER LOGIC - RESOLUTION
--------------------------------

Enter number of CNF clauses: 6

Enter clauses using | for OR.
Use ~ or NOT for negation.
Example: ~Food(x) | Likes(John,x)
Enter clause 1:  ~Food(x) | Likes(John,x)
Enter clause 2: ~Eats(x,y) | Killed(x) | Food(y)
Enter clause 3: Eats(Anil,Peanuts)
Enter clause 4: Alive(Anil)
Enter clause 5: ~Alive(x) | ~Killed(x)
Enter clause 6: ~Likes(John,Peanuts)

Starting Resolution...

Initial Clauses:
C1: ~Food(x) OR Likes(John,x)
C2: ~Eats(x,y) OR Killed(x) OR Food(y)
C3: Eats(Anil,Peanuts)
C4: Alive(Anil)
C5: ~Alive(x) OR ~Killed(x)
C6: ~Likes(John,Peanuts)

Resolution Process:

Step 1
Clause 1: ~Food(x) OR Likes(John,x)
Clause 2: ~Eats(x,y) OR Killed(x) OR Food(y)
New Clause: Likes(John,y_2) OR ~Eats(x_2,y_2) OR Killed(x_2)

Step 2
Clause 1: ~Food(x) OR Likes(John,x)
Clause 2: ~Likes(John,Peanuts)
New Clause: ~Food(Peanuts)

Step 3
Clause 1: ~Eats(x,y) OR Killed(x) OR Food(y)
Clause 2: Eats(Anil,Peanuts)
New Clause: Killed(Anil)